# Tableau - LOD
- https://help.tableau.com/current/pro/desktop/en-us/calculations_calculatedfields_lod.htm

## LODs in Tableau

### LOD - FIXED
    - create as calculated field
    - Create Calculate field by name of
    - LOD Customer Sales and put this notation in text box {FIXED [Customer Name] : SUM([Sales])}

### LOD Region Sales
    - {FIXED [Region] : SUM([Sales]) }

### LOD Customer Sales
- {FIXED [Customer Name] : SUM([Sales])}

### LOD Customer First Order
- { FIXED [Customer Name] : MIN([Order Date]) }

### How to use in Sheets
- Case-1
    - Part - 1
        - Rows - Customer Name
        - Columns - Sales : Sum(Sales)
    - Part -2
        - Add Region to Rows, next to customer name
        - This will show Region wise sales of each customer
        - Add LOD Customer Sales to Columns
        - This will add another dimension to the sheet showing total sales by customer
        - Compare Region Sales of each customer with its total Sales in all regions

In [ ]:
## Python - LOD

In [ ]:
# Data
import numpy as np
import pandas as pd
import datetime as dt

In [ ]:
url1 = 'https://raw.githubusercontent.com/dupadhyaya/piit/refs/heads/main/data/superstore_orders.csv'
print(url1)

In [ ]:
sales = pd.read_csv(url1)
sales.shape

In [ ]:
sales.head()

In [ ]:
sales.columns

In [ ]:
sales.info()

In [185]:
# sales by each customer
custTotal = sales[['Customer Name','Sales']].groupby('Customer Name')['Sales']\
    .agg('sum').round(0).reset_index().rename(columns= {'Sales':'Total'})
custTotal.head()

,Customer Name,Total
0,Aaron Bergman,886.00
1,Aaron Hawkins,"1,745.00"
2,Aaron Smayling,"3,051.00"
3,Adam Bellavance,"7,756.00"
4,Adam Hart,"3,250.00"


In [186]:
# Sort sales in decreasing order
custTotal.sort_values(by='Total', ascending=False).head(10)

,Customer Name,Total
693,Sean Miller,"25,043.00"
737,Tamara Chand,"19,052.00"
629,Raymond Buch,"15,117.00"
764,Tom Ashbrook,"14,596.00"
6,Adrian Barton,"14,474.00"
446,Ken Lonsdale,"14,175.00"
678,Sanjit Chand,"14,142.00"
336,Hunter Lopez,"12,873.00"
679,Sanjit Engle,"12,209.00"
157,Christopher Conant,"12,129.00"


In [187]:
#group Customer Sales by Region
custRegion = sales[['Customer Name','Region','Sales']].groupby(['Customer Name','Region'])['Sales'] \
    .agg('sum').round(0).reset_index()
custRegion.head()

,Customer Name,Region,Sales
0,Aaron Bergman,Central,577.00
1,Aaron Bergman,West,310.00
2,Aaron Hawkins,East,330.00
3,Aaron Hawkins,South,86.00
4,Aaron Hawkins,West,"1,328.00"


In [188]:
# join total Sales and region wise sales like LOD
custTotalRegion = pd.merge(custTotal, custRegion, on='Customer Name', how='inner')
custTotalRegion.shape

(2508, 4)

In [189]:
custTotalRegion.head()

,Customer Name,Total,Region,Sales
0,Aaron Bergman,886.00,Central,577.00
1,Aaron Bergman,886.00,West,310.00
2,Aaron Hawkins,"1,745.00",East,330.00
3,Aaron Hawkins,"1,745.00",South,86.00
4,Aaron Hawkins,"1,745.00",West,"1,328.00"


In [190]:
custTotalRegion.groupby('Customer Name').head()

,Customer Name,Total,Region,Sales
0,Aaron Bergman,886.00,Central,577.00
1,Aaron Bergman,886.00,West,310.00
2,Aaron Hawkins,"1,745.00",East,330.00
3,Aaron Hawkins,"1,745.00",South,86.00
4,Aaron Hawkins,"1,745.00",West,"1,328.00"
...,...,...,...,...
2503,Zuschuss Carroll,"8,026.00",South,"1,471.00"
2504,Zuschuss Carroll,"8,026.00",West,"2,641.00"
2505,Zuschuss Donatelli,"1,494.00",Central,331.00
2506,Zuschuss Donatelli,"1,494.00",South,857.00


In [191]:
## Index method
custTotal_idx = custTotal.set_index('Customer Name')
custRegion_idx = custRegion.set_index('Customer Name')
custRegion_idx.head()

,Region,Sales
Customer Name,,
Aaron Bergman,Central,577.00
Aaron Bergman,West,310.00
Aaron Hawkins,East,330.00
Aaron Hawkins,South,86.00
Aaron Hawkins,West,"1,328.00"


In [192]:
custTotalRegion = custTotal_idx.join( custRegion_idx, how='inner')
custTotalRegion.head()

,Total,Region,Sales
Customer Name,,,
Aaron Bergman,886.00,Central,577.00
Aaron Bergman,886.00,West,310.00
Aaron Hawkins,"1,745.00",East,330.00
Aaron Hawkins,"1,745.00",South,86.00
Aaron Hawkins,"1,745.00",West,"1,328.00"


- For pandas:
    * merge() → SQL-style joins on columns.
    * join() → Faster and cleaner when joining on indexes.
    * map() → Best when bringing a single aggregated value (like your customer total) into another DataFrame.

In [193]:
custRegion['Total'] = custRegion['Customer Name']\
    .map(custTotal.set_index('Customer Name')['Total'])
custRegion.sort_values(by='Total', ascending=False).head()

,Customer Name,Region,Sales,Total
2174,Sean Miller,West,10.00,"25,043.00"
2171,Sean Miller,Central,526.00,"25,043.00"
2172,Sean Miller,East,837.00,"25,043.00"
2173,Sean Miller,South,"23,669.00","25,043.00"
2315,Tamara Chand,West,7.00,"19,052.00"


In [194]:
#!pip install ipydatagrid
#from ipydatagrid import DataGrid
#DataGrid(custRegion)

In [195]:
#!pip install itables
from itables import show

In [196]:
show(custRegion)

Loading ITables v2.8.1 from the internet... (need help?)


In [197]:
GT(custRegion.head())

Customer Name,Region,Sales,Total
Aaron Bergman,Central,577.0,886.0
Aaron Bergman,West,310.0,886.0
Aaron Hawkins,East,330.0,1745.0
Aaron Hawkins,South,86.0,1745.0
Aaron Hawkins,West,1328.0,1745.0


In [198]:
## LOD - Minimum Order date
sales[['Customer Name', 'Order Date', 'Sales']].head(10)

,Customer Name,Order Date,Sales
0,Darren Powers,03/01/23,16.45
1,Phillina Ober,04/01/23,3.54
2,Phillina Ober,04/01/23,11.78
3,Phillina Ober,04/01/23,272.74
4,Mick Brown,05/01/23,19.54
5,Maria Etezadi,06/01/23,"2,573.82"
6,Maria Etezadi,06/01/23,5.48
7,Jack O'Briant,06/01/23,12.78
8,Maria Etezadi,06/01/23,609.98
9,Maria Etezadi,06/01/23,31.12


In [199]:
sales['Order Date']

0        03/01/23
1        04/01/23
2        04/01/23
3        04/01/23
4        05/01/23
           ...   
10189    30/12/26
10190    30/12/26
10191    30/12/26
10192    30/12/26
10193    30/12/26
Name: Order Date, Length: 10194, dtype: object

In [200]:
pd.to_datetime(sales['Order Date'],format='%d/%m/%y', dayfirst=True)

0       2023-01-03
1       2023-01-04
2       2023-01-04
3       2023-01-04
4       2023-01-05
           ...    
10189   2026-12-30
10190   2026-12-30
10191   2026-12-30
10192   2026-12-30
10193   2026-12-30
Name: Order Date, Length: 10194, dtype: datetime64[ns]

In [201]:
pd.to_datetime(sales['Order Date'],format='%d/%m/%y', dayfirst=True) #.dt.date

0       2023-01-03
1       2023-01-04
2       2023-01-04
3       2023-01-04
4       2023-01-05
           ...    
10189   2026-12-30
10190   2026-12-30
10191   2026-12-30
10192   2026-12-30
10193   2026-12-30
Name: Order Date, Length: 10194, dtype: datetime64[ns]

In [202]:
sales['Order Date'] = pd.to_datetime(sales['Order Date'],format='%d/%m/%y', \
                                     dayfirst=True).dt.normalize()
sales['Ship Date'] = pd.to_datetime(sales['Ship Date'],format='%d/%m/%y', \
                                     dayfirst=True).dt.normalize()
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          10194 non-null  int64         
 1   Order ID        10194 non-null  object        
 2   Order Date      10194 non-null  datetime64[ns]
 3   Ship Date       10194 non-null  datetime64[ns]
 4   Ship Mode       10194 non-null  object        
 5   Customer ID     10194 non-null  object        
 6   Customer Name   10194 non-null  object        
 7   Segment         10194 non-null  object        
 8   Country/Region  10194 non-null  object        
 9   City            10194 non-null  object        
 10  State/Province  10194 non-null  object        
 11  Postal Code     10194 non-null  object        
 12  Region          10194 non-null  object        
 13  Product ID      10194 non-null  object        
 14  Category        10194 non-null  object        
 15  Su

## Range of Date and numeric values to check version of data


In [203]:
pd.set_option('display.float_format', '{:,.2f}'.format)

In [204]:
# Numeric columns
num_range = sales.select_dtypes(include='number').agg(['min', 'max','sum']).T
# Date columns
date_range = sales.select_dtypes(include='datetime').agg(['min', 'max','count']).T
print("Numeric Columns")
display(num_range)
print("Date Columns")
display(date_range)

Numeric Columns


,min,max,sum
Row ID,1.00,"10,194.00","51,963,915.00"
Sales,0.44,"22,638.48","2,326,534.35"
Quantity,1.00,14.00,"38,654.00"
Discount,0.00,0.80,"1,583.99"
Profit,"-6,599.98","8,399.98","292,296.81"


Date Columns


,min,max,count
Order Date,2023-01-03 00:00:00,2026-12-30 00:00:00,10194
Ship Date,2023-01-07 00:00:00,2027-01-05 00:00:00,10194


In [ ]:
sales.describe()

In [ ]:
custFirstOrder = (sales.groupby('Customer Name', as_index=False)['Order Date'].min()\
    .rename(columns={'Order Date': 'First Order Date'}))
custFirstOrder[['Customer Name', 'First Order Date']].sort_values(by='First Order Date')

In [ ]:
custSummary = (sales.groupby('Customer Name', as_index=False).\
    agg(First_Order_Date =('Order Date','min'), Total_Sales = ('Sales','sum')))
custSummary.head()

In [ ]:
custSummary[['Customer Name', 'Total_Sales', 'First_Order_Date']].\
    sort_values(by=['Total_Sales', 'First_Order_Date'], ascending=[False,True])

## EndHere